# Data Profiling – Mobile Legends Play Store Dataset

Notebook ini digunakan untuk profiling awal dataset `mobilelegends.csv`.

Dataset berada di:
`data/raw/archive/mobilelegends.csv`

> Karena file CSV berukuran sekitar 3 GB, notebook menggunakan **chunking** agar tidak perlu memuat seluruh dataset ke RAM sekaligus.


## Struktur Dataset

Berdasarkan dataset yang kamu tunjukkan, kolomnya adalah:

| Kolom | Keterangan |
|---|---|
| `reviewId` | ID review |
| `userName` | Nama pengguna |
| `userImage` | URL gambar pengguna |
| `content` | Isi review |
| `score` | Rating review (1–5) |
| `thumbsUpCount` | Jumlah thumbs up |
| `reviewCreatedVersion` | Versi game saat review dibuat |
| `at` | Waktu/tanggal review |
| `replyContent` | Isi balasan pengembang |
| `repliedAt` | Waktu/tanggal balasan pengembang |

Unit analisis: **satu review pengguna Mobile Legends di Google Play Store**.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/raw/archive/mobilelegends.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH}. "
        "Pastikan notebook berada di folder notebooks/."
    )

file_size_gb = os.path.getsize(DATA_PATH) / (1024**3)
print(f"Lokasi dataset : {DATA_PATH}")
print(f"Ukuran file    : {file_size_gb:.2f} GB")


## 1. Preview Data

In [ ]:
preview = pd.read_csv(DATA_PATH, nrows=5)

print("Jumlah kolom:", len(preview.columns))
print("\nNama kolom:")
for i, col in enumerate(preview.columns, 1):
    print(f"{i}. {col}")

display(preview)


## 2. Tipe Data

In [ ]:
display(preview.dtypes.to_frame("dtype"))


## 3. Profiling Seluruh Dataset

Dataset diproses menggunakan `chunksize=50_000`.

Profiling mencakup:
- jumlah baris;
- missing values;
- `reviewId` unik dan duplikat;
- distribusi `score`;
- jumlah review yang memiliki balasan pengembang.


In [ ]:
CHUNK_SIZE = 50_000

row_count = 0
missing_counts = pd.Series(0, index=preview.columns, dtype="int64")
score_counts = pd.Series(dtype="int64")
unique_review_ids = set()
duplicate_review_ids = 0
replied_count = 0

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE):
    row_count += len(chunk)
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0)

    if "reviewId" in chunk.columns:
        for review_id in chunk["reviewId"].dropna():
            if review_id in unique_review_ids:
                duplicate_review_ids += 1
            else:
                unique_review_ids.add(review_id)

    if "score" in chunk.columns:
        score_counts = score_counts.add(chunk["score"].value_counts(), fill_value=0)

    if "replyContent" in chunk.columns:
        replied_count += chunk["replyContent"].notna().sum()

print(f"Total baris: {row_count:,}")
print(f"Total kolom: {len(preview.columns)}")
print(f"Review ID unik: {len(unique_review_ids):,}")
print(f"Review ID duplikat: {duplicate_review_ids:,}")
print(f"Review dengan balasan pengembang: {replied_count:,}")


## 4. Missing Values

In [ ]:
missing_table = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.astype(int).values,
    "missing_percent": (missing_counts / row_count * 100).round(2).values
}).sort_values("missing_count", ascending=False)

display(missing_table)


## 5. Distribusi Rating

In [ ]:
score_counts = score_counts.sort_index().astype(int)

score_table = pd.DataFrame({
    "score": score_counts.index.astype(int),
    "count": score_counts.values,
    "percentage": (score_counts.values / row_count * 100).round(2)
})

display(score_table)

plt.figure(figsize=(8, 5))
plt.bar(score_table["score"].astype(str), score_table["count"])
plt.xlabel("Rating (score)")
plt.ylabel("Jumlah Review")
plt.title("Distribusi Rating Review Mobile Legends")
plt.tight_layout()
plt.show()


## 6. Statistik Deskriptif

In [ ]:
numeric_frames = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=lambda c: c in ["score", "thumbsUpCount"],
    chunksize=CHUNK_SIZE
):
    numeric_frames.append(chunk)

numeric_data = pd.concat(numeric_frames, ignore_index=True)
display(numeric_data.describe().T)


## 7. Periode Data

In [ ]:
min_date = None
max_date = None

for chunk in pd.read_csv(DATA_PATH, usecols=["at"], chunksize=CHUNK_SIZE):
    dates = pd.to_datetime(chunk["at"], errors="coerce")
    chunk_min = dates.min()
    chunk_max = dates.max()

    if pd.notna(chunk_min):
        min_date = chunk_min if min_date is None else min(min_date, chunk_min)
    if pd.notna(chunk_max):
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

print("Tanggal review paling awal :", min_date)
print("Tanggal review paling akhir:", max_date)


## 8. Kesimpulan Profiling

Setelah seluruh cell dijalankan, gunakan hasil di atas untuk menjelaskan:

1. ukuran dataset dan jumlah baris/kolom;
2. tipe data setiap kolom;
3. kondisi missing values;
4. potensi duplikasi berdasarkan `reviewId`;
5. distribusi rating `score`;
6. jumlah review yang memperoleh balasan pengembang; dan
7. periode waktu review.

Penggunaan `chunksize` menunjukkan bahwa dataset besar dapat diproses bertahap tanpa harus memuat seluruh file 3 GB ke memori.
